In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Define paths
import os
DATASET_DIR = "/content/drive/MyDrive/Pinterest Interior Design Images and Metadata"
IMAGE_DIR = os.path.join(DATASET_DIR, "images")
metadata_csv = os.path.join(DATASET_DIR, "metadata.csv")
train_csv = os.path.join(DATASET_DIR, "train_data.csv")
val_csv = os.path.join(DATASET_DIR, "val_data.csv")
test_csv = os.path.join(DATASET_DIR, "test_data.csv")
import pandas as pd
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)
for df in [train_df, val_df, test_df]:
    df['image_path'] = df['image_path'].str.replace(r'^(\.\./)*data/raw/', '', regex=True)

print("Image paths cleaned")

Image paths cleaned


# **Exploration**

In [ ]:
IMAGE_DIR = DATASET_DIR

os.path.exists(os.path.join(IMAGE_DIR, train_df['image_path'].iloc[0]))

True

In [ ]:
# Display column names in a list
print("Data columns:", train_df.columns.tolist())

train_df.shape

print("\nGeneral structure:")
print(train_df.info())


print("\nFirst 5 rows:")
print(train_df.head())


Data columns: ['image_path', 'room_type', 'style', 'encoded_label', 'full_path']

General structure:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   image_path     13 non-null     object
 1   room_type      13 non-null     object
 2   style          13 non-null     object
 3   encoded_label  13 non-null     int64 
 4   full_path      13 non-null     object
dtypes: int64(1), object(4)
memory usage: 652.0+ bytes
None

First 5 rows:
                                       image_path room_type       style  \
0               kitchen/boho/kitchen_boho_171.jpg   kitchen        boho   
1            kitchen/modern/kitchen_modern_67.jpg   kitchen      modern   
2  bathroom/minimalist/bathroom_minimalist_54.jpg  bathroom  minimalist   
3            bedroom/modern/bedroom_modern_98.jpg   bedroom      modern   
4            bedroom/modern/bedroom_modern_74.

In [ ]:
print(f"\nNumber of training images: {len(train_df)}")
print(f"Room type classes: {train_df['room_type'].nunique()} ----> {sorted(train_df['room_type'].unique())}")
print(f"Style classes: {train_df['style'].nunique()} ----> {sorted(train_df['style'].unique())}")


Number of training images: 13
Room type classes: 4 ----> ['bathroom', 'bedroom', 'kitchen', 'living_room']
Style classes: 5 ----> ['boho', 'industrial', 'minimalist', 'modern', 'scandinavian']


In [ ]:
# بعد قراءة بياناتك وتقسيمها (train_df, val_df, test_df)
from sklearn.preprocessing import LabelEncoder

# Encode the labels
style_encoder = LabelEncoder()
style_encoder.fit(train_df['style'])  # أو df['style'] على حسب dataframe

class_names = list(style_encoder.classes_)  # تعريف class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Count number of samples per class in train set
class_counts = train_df['style'].value_counts().sort_index()

print("Number of images per class in the training set:")
for cls, count in class_counts.items():
    print(f"{cls}: {count} images")


Classes: ['boho', 'industrial', 'minimalist', 'modern', 'scandinavian']
Number of classes: 5
Number of images per class in the training set:
boho: 3 images
industrial: 3 images
minimalist: 2 images
modern: 4 images
scandinavian: 1 images


In [ ]:
os.path.exists(os.path.join(IMAGE_DIR, train_df['image_path'].iloc[0]))

True

In [ ]:
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

# Save CSVs
train_df.to_csv(os.path.join(DATASET_DIR, "train_data.csv"), index=False)
val_df.to_csv(os.path.join(DATASET_DIR, "val_data.csv"), index=False)
test_df.to_csv(os.path.join(DATASET_DIR, "test_data.csv"), index=False)




print(train_df.columns)
print(val_df.columns)
print(test_df.columns)

paths = df['full_path'].values

Train size: 13
Validation size: 5
Test size: 1
Index(['image_path', 'room_type', 'style', 'encoded_label', 'full_path'], dtype='object')
Index(['image_path', 'room_type', 'style', 'encoded_label', 'full_path'], dtype='object')
Index(['image_path', 'room_type', 'style', 'encoded_label', 'full_path'], dtype='object')


In [ ]:
def create_tf_dataset(image_dir, df, img_size=(224, 224), batch_size=32, training=False):

    def load_preprocess(image_path, label):
        # Build correct full path
        full_path = tf.strings.join([image_dir, "/", image_path])

        # Load and preprocess
        image = tf.io.read_file(full_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, img_size)
        image = efficientnet.preprocess_input(image)

        return image, label

    # Create TF dataset
    ds_paths = tf.data.Dataset.from_tensor_slices(df["image_path"].values)
    ds_labels = tf.data.Dataset.from_tensor_slices(df["style_encoded"].values)

    ds = tf.data.Dataset.zip((ds_paths, ds_labels))
    ds = ds.map(load_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=1000)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224,224,3)
)

image = preprocess_input(IMAGE_DIR)

In [ ]:
!pip install tensorflow --quiet




import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0 as efficientnet

IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

# Read metadata (same file used in your previous preprocessing steps)
df = pd.read_csv(os.path.join(DATASET_DIR, "metadata.csv"))

# Remove any invalid entries
df = df.dropna(subset=['style', 'image_path'])

# Full path for each image
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Classes
classes = sorted(df['style'].unique())
label2idx = {c: i for i, c in enumerate(classes)}

print("Classes:", classes)

# ----------- Augmentation + Preprocessing -------------
def preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # scale to [0,1]
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # Optional augmentation
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)

    # One-hot encode labels
    return img, tf.one_hot(label, len(classes))

# ----------- TF Dataset Builder -------------
def make_dataset(df, shuffle=False, augment=False):
    paths = df['full_path'].values
    labels = df['style'].map(label2idx).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)

    ds = ds.map(lambda p, l: preprocess(p, l, augment),
                num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH).prefetch(AUTOTUNE)
    return ds

# ----------- Build Datasets -------------
train_ds = make_dataset(train_df, shuffle=True, augment=True)
val_ds   = make_dataset(val_df,   shuffle=False, augment=False)
test_ds  = make_dataset(test_df,  shuffle=False, augment=False)

print("TF Datasets created successfully!")

Classes: ['boho', 'industrial', 'minimalist', 'modern', 'scandinavian']
TF Datasets created successfully!


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

# افترض أن class_names موجودة مسبقًا (من preprocessing)
num_classes = len(class_names)  # <- مهم تعريفه قبل استخدامه

# Base model (without top)
base_model = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')
base_model.trainable = False  # freeze base

# Build model
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)  # helps avoid overfitting
output = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_10        │ (None, 224, 224,  │          0 │ input_layer_5[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_5     │ (None, 224, 224,  │          7 │ rescaling_10[0][… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_11        │ (None, 224, 224,  │          0 │ normalization_5[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_11[0][… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

# **Data Augmentation & Pipeline**

In [ ]:
!pip install tensorflow --quiet

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, optimizers

# --- 1) Build the model ---
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
output = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# تنظيف مسارات الصور
df['image_path'] = df['image_path'].str.replace(r'^(\.\./)*data/raw/', '', regex=True)
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))
df['full_path'] = df['full_path'].str.replace("\\", "/")  # للتأكد من استخدام '/' بدل '\'


df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)

In [ ]:
IMG_SIZE = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

# Read metadata (same file used in your previous preprocessing steps)
df = pd.read_csv(os.path.join(DATASET_DIR, "metadata.csv"))

# Remove any invalid entries
df = df.dropna(subset=['style', 'image_path'])

# Full path for each image
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Classes
classes = sorted(df['style'].unique())
label2idx = {c: i for i, c in enumerate(classes)}

print("Classes:", classes)

# ----------- Augmentation + Preprocessing -------------
def preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # scale to [0,1]
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])

    # Optional augmentation
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)

    # One-hot encode labels
    return img, tf.one_hot(label, len(classes))

# ----------- TF Dataset Builder -------------
def make_dataset(df, shuffle=False, augment=False):
    paths = df['full_path'].values
    labels = df['style'].map(label2idx).values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=42)

    ds = ds.map(lambda p, l: preprocess(p, l, augment),
                num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH).prefetch(AUTOTUNE)
    return ds

# ----------- Build Datasets -------------
train_ds = make_dataset(train_df, shuffle=True, augment=True)
val_ds   = make_dataset(val_df,   shuffle=False, augment=False)
test_ds  = make_dataset(test_df,  shuffle=False, augment=False)

print("TF Datasets created successfully!")

Classes: ['boho', 'industrial', 'minimalist', 'modern', 'scandinavian']
TF Datasets created successfully!


#Build CNN (Transfer Learning: EfficientNetB0)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0


num_classes = len(class_names)

# Base model (without top)
base_model = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')
base_model.trainable = False  # freeze base

# Build model
x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)  # helps avoid overfitting
output = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_14        │ (None, 224, 224,  │          0 │ input_layer_7[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_7     │ (None, 224, 224,  │          7 │ rescaling_14[0][… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_15        │ (None, 224, 224,  │          0 │ normalization_7[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_15[0][… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,055,976 (15.47 MB)

 Trainable params: 6,405 (25.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

# **Train Model**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, optimizers

# --- 1) Build the model ---
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
output = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# Evaluation & Fine-tuning

In [ ]:
# تنظيف مسارات الصور
df['image_path'] = df['image_path'].str.replace(r'^(\.\./)*data/raw/', '', regex=True)
df['full_path'] = df['image_path'].apply(lambda x: os.path.join(IMAGE_DIR, x))
df['full_path'] = df['full_path'].str.replace("\\", "/")  # للتأكد من استخدام '/' بدل '\'

In [ ]:
df = df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np

# ============================
# 0) Paths and parameters
# ============================

DATASET_PATH = "/content/drive/MyDrive/Pinterest Interior Design Images and Metadata"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# ============================
# 1) Load Dataset (raw)
# ============================

raw_train_ds = image_dataset_from_directory(
    DATASET_PATH,
    labels="inferred",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="training",
    seed=42,
    shuffle=True
)

raw_val_ds = image_dataset_from_directory(
    DATASET_PATH,
    labels="inferred",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    subset="validation",
    seed=42,
    shuffle=False
)

# Get class names before map/prefetch
class_names = raw_train_ds.class_names
num_classes = len(class_names)
print("Classes found:", class_names)

# ============================
# 2) Data Augmentation
# ============================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = raw_train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = raw_val_ds.prefetch(buffer_size=AUTOTUNE)

# ============================
# 3) Build Improved Model
# ============================

def build_improved_model(num_classes):
    inputs = tf.keras.Input(shape=(224, 224, 3))

    # Pretrained EfficientNetB0
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        input_tensor=inputs,
        pooling='avg',
        weights='imagenet'
    )
    base_model.trainable = True  # fine-tune all layers

    x = layers.Dropout(0.4)(base_model.output)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

model = build_improved_model(num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============================
# 4) Callbacks
# ============================

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.h5", monitor='val_accuracy', save_best_only=True
    )
]

# ============================
# 5) Train the Model
# ============================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=callbacks
)

# ============================
# 6) Evaluate on Validation Set
# ============================

val_loss, val_acc = model.evaluate(val_ds)
print(f"\n✅ Validation Accuracy: {val_acc*100:.2f}%")

# ============================
# 7) Optional: Plot training history
# ============================

plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


Found 4142 files belonging to 4 classes.
Using 3314 files for training.
Found 4142 files belonging to 4 classes.
Using 828 files for validation.
Classes found: ['bathroom', 'bedroom', 'kitchen', 'living_room']


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_12        │ (None, 224, 224,  │          0 │ input_layer_7[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_6     │ (None, 224, 224,  │          7 │ rescaling_12[0][… │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_13        │ (None, 224, 224,  │          0 │ normalization_6[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_13[0][… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,707,495 (17.96 MB)

 Trainable params: 4,665,472 (17.80 MB)

...

Epoch 1/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.3704 - loss: 1.3118

104/104 ━━━━━━━━━━━━━━━━━━━━ 648s 6s/step - accuracy: 0.3713 - loss: 1.3106 - val_accuracy: 0.6848 - val_loss: 0.7524 - learning_rate: 1.0000e-04
Epoch 2/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 633s 6s/step - accuracy: 0.6294 - loss: 0.9065 - val_accuracy: 0.5761 - val_loss: 0.9687 - learning_rate: 1.0000e-04
Epoch 3/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 622s 6s/step - accuracy: 0.6952 - loss: 0.7793 - val_accuracy: 0.6159 - val_loss: 0.9009 - learning_rate: 1.0000e-04
Epoch 4/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.7359 - loss: 0.6897

104/104 ━━━━━━━━━━━━━━━━━━━━ 598s 6s/step - accuracy: 0.7358 - loss: 0.6897 - val_accuracy: 0.6884 - val_loss: 0.7483 - learning_rate: 1.0000e-04
Epoch 5/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7563 - loss: 0.6219

104/104 ━━━━━━━━━━━━━━━━━━━━ 700s 7s/step - accuracy: 0.7562 - loss: 0.6220 - val_accuracy: 0.7440 - val_loss: 0.6924 - learning_rate: 1.0000e-04
Epoch 6/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 710s 7s/step - accuracy: 0.7766 - loss: 0.5709 - val_accuracy: 0.7101 - val_loss: 0.7377 - learning_rate: 1.0000e-04
Epoch 7/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.7828 - loss: 0.5400

104/104 ━━━━━━━━━━━━━━━━━━━━ 744s 7s/step - accuracy: 0.7829 - loss: 0.5400 - val_accuracy: 0.7923 - val_loss: 0.5592 - learning_rate: 1.0000e-04
Epoch 8/25
104/104 ━━━━━━━━━━━━━━━━━━━━ 705s 7s/step - accuracy: 0.8169 - loss: 0.4909 - val_accuracy: 0.7041 - val_loss: 0.7609 - learning_rate: 1.0000e-04
Epoch 9/25
  5/104 ━━━━━━━━━━━━━━━━━━━━ 10:25 6s/step - accuracy: 0.8381 - loss: 0.5283

# **Deployment**

In [ ]:
# app_simple.py
# Gradio Deployment for Interior Design Style Classifier

import os
import numpy as np
import tensorflow as tf
from PIL import Image
import gradio as gr

# =========================
# 1) Load Model
# =========================
MODEL_PATH = "/content/best_model.h5"  # Update path if needed

# Load the model
if not os.path.exists(MODEL_PATH):
    # If 'model' variable exists in notebook, save it
    model.save(MODEL_PATH)

model = tf.keras.models.load_model(MODEL_PATH)

# =========================
# 2) Classes
# =========================
# Ensure class_names is defined in your notebook
# Example: class_names = ["Modern", "Industrial", "Scandinavian", ...]
CLASS_NAMES = class_names  # from your notebook preprocessing

# =========================
# 3) Prediction Function
# =========================
def predict_image(img):
    """
    Predict the style of the uploaded interior design image.
    Returns only the top predicted class (word).
    """
    img = img.convert("RGB")
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)[0]

    # Get top predicted class
    top_idx = int(np.argmax(predictions))
    top_class = CLASS_NAMES[top_idx]

    return top_class  # Return as word only

# =========================
# 4) Gradio Interface
# =========================
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Interior Design Image"),
    outputs=gr.Text(label="Predicted Style"),
    title="Interior Design Style Classifier",
    description="Upload an interior design photo to classify its style.",
)

# Launch App
interface.launch(share=True)
